In [ ]:
import pandas as pd
import numpy as np
import os

# 1. Đường dẫn thư mục của bạn
WORK_DIR = r"E:\nhập môn AI\student_data"
SILVER_DIR = r"E:\nhập môn AI\student_data\sivler"
os.makedirs(SILVER_DIR, exist_ok=True)

def clean_and_process_promotions():
    print("1. Đang xử lý promotions.csv...")
    file_path = os.path.join(WORK_DIR, "promotions.csv")
    df = pd.read_csv(file_path)
    
    # Xử lý dữ liệu trùng lặp
    df = df.drop_duplicates(subset=['promo_id'], keep='first')
    
    # Xử lý dữ liệu khuyết/lỗi (40 dòng applicable_category bị null)
    df['applicable_category'] = df['applicable_category'].fillna('All')
    
    # Đảm bảo đúng schema
    target_cols = ['promo_id', 'promo_name', 'promo_type', 'discount_value', 'start_date', 
                   'end_date', 'applicable_category', 'promo_channel', 'stackable_flag', 'min_order_value']
    df_silver = df[target_cols]
    
    output_path = os.path.join(SILVER_DIR, "promotions.csv")
    df_silver.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"   -> Đã xử lý xong và lưu: {output_path}")

def clean_and_process_inventory():
    print("2. Đang xử lý inventory.csv...")
    file_path = os.path.join(WORK_DIR, "inventory.csv")
    df = pd.read_csv(file_path)
    
    # Xóa trùng lặp dựa trên khóa chính
    df = df.drop_duplicates(subset=['snapshot_date', 'product_id'], keep='first')
    
    # Xử lý giá trị âm (không nhất quán nếu có)
    df['stock_on_hand'] = df['stock_on_hand'].apply(lambda x: max(x, 0))
    df['units_sold'] = df['units_sold'].apply(lambda x: max(x, 0))

    # TÍNH TOÁN CÁC CỜ (SILVER DATA)
    df['stockout_flag'] = np.where(df['stock_on_hand'] <= 0, 1, 0)
    df['reorder_flag'] = np.where((df['stock_on_hand'] > 0) & (df['stock_on_hand'] < 20) & (df['units_sold'] > 0), 1, 0)
    df['overstock_flag'] = np.where(df['stock_on_hand'] > 500, 1, 0)
    
    # CHUẨN HÓA 2NF/3NF: Bỏ các cột rác (product_name, category, segment, year, month)
    target_cols = ['snapshot_date', 'product_id', 'stock_on_hand', 'units_received', 'units_sold', 
                   'stockout_days', 'days_of_supply', 'fill_rate', 'stockout_flag', 
                   'overstock_flag', 'reorder_flag', 'sell_through_rate']
    
    df_silver = df[target_cols]
    output_path = os.path.join(SILVER_DIR, "inventory.csv")
    df_silver.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"   -> Đã chuẩn hóa 2NF/3NF và lưu: {output_path}")

def clean_and_process_order_items():
    print("3. Đang xử lý order_items.csv...")
    file_path = os.path.join(WORK_DIR, "order_items.csv")
    
    # Dùng low_memory=False để tránh lỗi mixed types từ dữ liệu rác
    df = pd.read_csv(file_path, low_memory=False)
    
    # Phát hiện và xóa 16 dòng trùng lặp khóa chính (order_id, product_id)
    duplicates_count = df.duplicated(subset=['order_id', 'product_id']).sum()
    if duplicates_count > 0:
        print(f"   * Đã dọn dẹp {duplicates_count} dòng vi phạm trùng lặp khóa chính.")
    df = df.drop_duplicates(subset=['order_id', 'product_id'], keep='first')
    
    # Xử lý dữ liệu khuyết/NULL
    df['promo_id'] = df['promo_id'].fillna('None')
    if 'promo_id_2' in df.columns:
        df['promo_id_2'] = df['promo_id_2'].fillna('None')
        
    # Sửa lỗi logic: Số lượng và giá không được âm
    df['quantity'] = df['quantity'].apply(lambda x: max(x, 1)) # Bán ít nhất 1 sản phẩm
    df['unit_price'] = df['unit_price'].apply(lambda x: max(x, 0))

    target_cols = ['order_id', 'product_id', 'quantity', 'unit_price', 'discount_amount', 'promo_id', 'promo_id_2']
    df_silver = df[target_cols]
    
    output_path = os.path.join(SILVER_DIR, "OrderItems.csv")
    df_silver.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"   -> Đã xử lý xong và lưu: {output_path}")

if __name__ == "__main__":
    print(f"BẮT ĐẦU DỌN DẸP, CHUẨN HÓA VÀ LƯU VÀO: {SILVER_DIR}\n")
    try:
        clean_and_process_promotions()
        clean_and_process_inventory()
        clean_and_process_order_items()
        print("\n Dữ liệu của bạn hiện tại đã 'sạch' 100% và đạt chuẩn.")
    except Exception as e:
        print(f"Có lỗi xảy ra, vui lòng kiểm tra lại file: {e}")

BẮT ĐẦU DỌN DẸP, CHUẨN HÓA VÀ LƯU VÀO: E:\nhập môn AI\student_data\sivler

1. Đang xử lý promotions.csv...
   -> Đã xử lý xong và lưu: E:\nhập môn AI\student_data\sivler\promotions.csv
2. Đang xử lý inventory.csv...
   -> Đã chuẩn hóa 2NF/3NF và lưu: E:\nhập môn AI\student_data\sivler\inventory.csv
3. Đang xử lý order_items.csv...
   * Đã dọn dẹp 16 dòng vi phạm trùng lặp khóa chính.
   -> Đã xử lý xong và lưu: E:\nhập môn AI\student_data\sivler\OrderItems.csv

HOÀN TẤT XUẤT SẮC! Dữ liệu của bạn hiện tại đã 'sạch' 100% và đạt chuẩn.
